# Sentiment Classification Project

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
EMBEDDING_DIR = PROJECT_ROOT / "experiments" / "embeddings"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [1]:
import sys
print(sys.executable)

/cluster/courses/cil/envs/envs/text-classification/bin/python


In [1]:
import sys, torch, socket
print("host:", socket.gethostname())
print("python:", sys.executable)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")

host: studgpu-node01.inf.ethz.ch
python: /cluster/courses/cil/envs/envs/text-classification/bin/python
torch: 2.11.0+cu126
cuda available: True
cuda version: 12.6
device: NVIDIA GeForce RTX 5060 Ti


/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 NVIDIA GeForce RTX 5060 Ti which is of compute capability (CC) 12.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 5.0 which supports hardware CC >=5.0,<6.0 except {5.3}
- 6.0 which supports hardware CC >=6.0,<7.0 except {6.2}
- 7.0 which supports hardware CC >=7.0,<8.0 except {7.2}
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.8, 13.0
  _warn_unsupported_code(d, device_cc, code_ccs)
/cluster/courses/cil/envs/envs/text-classification/lib/python3.14/site-packages/torch/cuda/__init__.py:489: UserWa

In [2]:
import torch
x = torch.randn(1000, 1000, device="cuda")
y = x @ x
print(y.mean().item())

AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# Load data

In [ ]:
train_full = pd.read_csv(DATA_DIR / "train.csv")
print(train_full[5:10])
print("\n\n")
print("-" * 50)
print("\n\n")
print(train_full["sentence"].iloc[6])

# Build Validation Set
We use 90% of the reviews for training, and the remaining 10% for validation

In [ ]:
train_df, val_df = train_test_split(
        train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)

# Bag-of-words + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# We only keep the 10'000 most frequent words and bigrams (i.e. word pairs)
# This is both to reduce the computational cost and reduce potential overfitting
vectorizer = CountVectorizer(ngram_range=(1, 2), max_features=10000)

In [ ]:
# Important: Fit ONLY on training data
X_train = vectorizer.fit_transform(train_df["sentence"])
X_val = vectorizer.transform(val_df["sentence"])

Y_train = train_df["label"]
Y_val = val_df["label"]

In [ ]:
X_train

In [ ]:
print(X_train[2])

Now we train a logistic regression classifier...

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
# Logistic Regression Classifier, C: Inverse of regularization strength , max_iter: Maximum number of training iterations
model = LogisticRegression(C=1.0, max_iter=100)
model.fit(X_train, Y_train)

# Evaluate model

In [ ]:
Y_train_pred = model.predict(X_train)
Y_val_pred = model.predict(X_val)

In [ ]:
from sklearn.metrics import mean_absolute_error

In [ ]:
# score on training set
mae_train = mean_absolute_error(Y_train, Y_train_pred)
score_train = 1.0 - (mae_train / 4.0)
accuracy_train = np.mean(Y_train == Y_train_pred)

# score on validation set
mae_val = mean_absolute_error(Y_val, Y_val_pred)
score_val = 1.0 - (mae_val / 4.0)
accuracy_val = np.mean(Y_val == Y_val_pred)
    

print(f"Training Score: {score_train:.4f}, MAE: {mae_train:.4f}, Accuracy: {accuracy_train:.4f}")
print(f"Validation Score: {score_val:.4f}, MAE: {mae_val:.4f}, Accuracy: {accuracy_val:.4f}")

# Make test submission

In [ ]:
test_df = pd.read_csv(DATA_DIR / "test.csv")
X_test = vectorizer.transform(test_df["sentence"])

In [ ]:
submit_preds = model.predict(X_test)
submission = pd.DataFrame({
    "id": test_df["id"],
    "label": submit_preds
})

submission_path = "submissions/submission.csv"
submission.to_csv(submission_path, index=False)

# Model Interpretation

In [ ]:
# Top N most Important Words & Word Pairs per Output Class (Pos, Neutral, Negative)
feature_names = vectorizer.get_feature_names_out() # get names of all tokens from vectorizer
coefs = model.coef_  # Weights per Feature for each Output Class; Shape: (Num_Output_Classes, Num_Features)

# Get Top_n Features by Weight for each Class
def get_top_features(class_index, top_n=10):
    class_coef = coefs[class_index]
    top_indices = np.argsort(class_coef)[-top_n:]
    return [feature_names[i] for i in reversed(top_indices)]

print("Top words & bigrams for 1 stars:", get_top_features(0))
print("Top words & bigrams for 2 star:", get_top_features(1))
print("Top words & bigrams for 3 stars:", get_top_features(2))
print("Top words & bigrams for 4 stars:", get_top_features(3))
print("Top words & bigrams for 5 stars:", get_top_features(4))

In [ ]:
# Confusion Matrix - Negative, Neutral, Positive
from sklearn.metrics import confusion_matrix

conf_matrix = confusion_matrix(Y_val,Y_val_pred, labels=[0,1,2,3,4])
print(conf_matrix)

In [ ]:
results_df = val_df.copy()
results_df['predicted_label'] = Y_val_pred

results_df['diff'] = (results_df['label'] - results_df['predicted_label']).abs()
very_off = results_df[results_df['diff'] >= 3].sort_values(by='diff', ascending=False)

for i, row in very_off.head(10).iterrows():
    print(f"--- Review Snippet ---")
    print(f"{row['sentence'][:300]}...") # Increased to 300 to see more context
    print(f"Actual: {row['label']} | Predicted: {row['predicted_label']}")
    print("-" * 30 + "\n")